In [31]:
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib qt
import random as r
from collections import deque

In [41]:
def sim_instance(
    simlength,
    mrtakethatriskvalue= 0.2,
    alpha= 0.5,
    gamma= 0.5,
    t_hbins = 100,
    ideal_h= 0.5,
    h_mu = 0.05,
    h_si = 0.03,
    hmax= 1,
    drink_amount = 0.09,
    t_dbins = 10,
    effect_length = 10,
    do_live_plot=False,
    batch_run = False
):
    
    h = ideal_h
    h_bin_spread = []

    max_incd_l = []

    eval_boundary = simlength - 2000

    eval_h = []
    eval_l = []
    eval_d = np.array([])

    if batch_run:
        do_live_plot = False

    def curr_loss(cur_h):
        l = (cur_h - ideal_h) ** 2 # bro really said abs...
        return l

    Q_table = np.zeros((t_hbins, t_dbins, 2)) # pairs in each grid square > val of wait: Q_table[hbin, incdbin, 0]; val of drink: Q_table[hbin, incdbin, 1]

    if do_live_plot:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))  # one window, two rows

        # hydration (top)
        ax1.set_xlim(0, simlength)
        line1, = ax1.plot([], [], lw=1)
        dot1,  = ax1.plot([], [], "o", c="blue")
        ax1.set_ylabel("hydration")

        # loss (bottom)
        ax2.set_xlim(0, simlength)
        ax2.axhline(0.01, ls="--", c="red")
        line2, = ax2.plot([], [], lw=1)
        dot2,  = ax2.plot([], [], "o", c="red")
        ax2.set_ylabel("loss")

        xs, ys1, ys2 = [], [], []

    carry_hbin = int(h * t_hbins / hmax) # starting bin
    carry_hbin = min(t_hbins - 1, carry_hbin)

    carry_incdbin = 0

    carry_incdbin = int(0 * t_dbins / (0.3)) # bin intial
    carry_incdbin = min(t_dbins - 1, carry_incdbin)

    carry_action = 0
    carry_loss = 0

    action_que = deque(maxlen=effect_length)


    # SIM LOOP START HERE
    for t in range(0, simlength):  

        #print("tick starting:", t)

        h -= np.random.normal(h_mu, h_si)
        h = max(0, h)

        for pos, act in enumerate(action_que):
            if act == 1:
                if pos == 2:
                    h += 0.5 * drink_amount
                elif pos == 4:
                    h += 0.25 * drink_amount
                elif pos == 7:
                    h += 0.25 * drink_amount
        h_bin = int(h * t_hbins / hmax)
        h_bin = min(t_hbins - 1, h_bin)
        h_bin_spread.append(h_bin)

        inc_drink = 0
        for pos, act in enumerate(action_que):
            if act == 1:
                if pos < 2:
                    inc_drink += 0.5 * drink_amount
                elif pos < 4:
                    inc_drink += 0.25 * drink_amount
                elif pos < 7:
                    inc_drink += 0.25 * drink_amount
        incd_bin = int(inc_drink * t_dbins / (0.3)) # bin based on observations
        incd_bin = min(t_dbins - 1, incd_bin)

        #print(action_que)
        max_incd_l.append(inc_drink)

        if t != 0 and t < eval_boundary: # table update if we still learning
            Q_table[carry_hbin, carry_incdbin, carry_action] += alpha * (carry_loss + (gamma * Q_table[h_bin, incd_bin].min()) - Q_table[carry_hbin, carry_incdbin, carry_action]) # memeory nudged by the error our new observation makes to it

        if do_live_plot:
            xs.append(t)

        if r.random() < mrtakethatriskvalue and t < eval_boundary:
                if r.random() < 0.5:
                    action_que.appendleft(1)
                else:
                    action_que.appendleft(0)
                    
        else:
            action_que.appendleft(int(np.argmin(Q_table[h_bin, incd_bin]))) # at this state, what do we think is the best move?
            
        l = curr_loss(h) # CALC LOSS HERE FOR TIMESTEP T

        carry_hbin = h_bin
        carry_incdbin = incd_bin
        carry_action = action_que[0]
        carry_loss = l
            
        if do_live_plot:
            ys1.append(h)
            ys2.append(l)

            zoom_start = int(0.9 * simlength)
            draw_every = simlength / 20

            should_draw = (t < zoom_start and t % draw_every == 0) or (t >= zoom_start)

            if should_draw:
                line1.set_data(xs, ys1)
                dot1.set_data([t], [h])

                line2.set_data(xs, ys2)
                dot2.set_data([t], [l])

                if t < zoom_start:
                    ax1.set_ylim(0, max(0.011, max(ys1) * 1.1))
                    ax2.set_ylim(0, max(0.011, max(ys2) * 1.1))
                else:
                    recent_h = ys1[zoom_start:]
                    recent_l = ys2[zoom_start:]

                    max_dev = max(abs(y - ideal_h) for y in recent_h)
                    half_range = max(0.03, max_dev * 1.2)

                    ax1.set_ylim(ideal_h - half_range, ideal_h + half_range)
                    ax2.set_ylim(0, max(0.011, max(recent_l) * 1.1))

                    window = 200  # show last 200 ticks
                    ax1.set_xlim(max(0, t - window), t)
                    ax2.set_xlim(max(0, t - window), t)

                    ax1.axvline(eval_boundary, ls=':', c='gray')   # dotted vertical at freeze point
                    ax2.axvline(eval_boundary, ls=':', c='gray')   # dotted vertical at freeze point


                ax1.axhline(ideal_h, ls="-", c="pink")   # the ideal hydration cenetered
                fig.canvas.draw_idle()
                plt.pause(0.1)
                
        if t >= eval_boundary:
            eval_h.append(h)
            eval_l.append(l)
            # eval_d(abs(ideal_h - h)) do later when vectorised!

    eval_h = np.array(eval_h)
    eval_l = np.array(eval_l)
    eval_d = np.abs(ideal_h - eval_h)
    max_incd_l = np.array(max_incd_l)

    return {
        "mean_loss": eval_l.mean(),
        "mean_dist": eval_d.mean(),
        "min_loss": eval_l.min(),
        "max_incd": max_incd_l[eval_boundary: ].max(),
        "h_bin_spread": h_bin_spread
    }


In [49]:
s = 20000

h_bin_spread = sim_instance(s, do_live_plot=False, hmax=2, t_hbins=50, mrtakethatriskvalue=0.05).get("h_bin_spread")

max_bin = np.array(h_bin_spread).max()

bins = [i for i in range(max_bin + 1)]

plt.hist(h_bin_spread, bins=bins, edgecolor="black")
plt.xlabel("h bin")
plt.ylabel("Frequency")
plt.title("Distribution of h bins")
plt.show()

In [ ]:
S = 10000
N = 30

alphas   = np.linspace(0, 1, 11)   # 0.00 to 1.00 51 "bins" so one every 0.02
epsilons = np.linspace(0, 1, 11)
n_eps, n_alpha = len(epsilons), len(alphas)

mean_loss = np.empty((n_eps, n_alpha, N)) # table with N layers
mean_dist = np.empty((n_eps, n_alpha, N))
min_loss  = np.empty((n_eps, n_alpha, N))
max_incd  = np.empty((n_eps, n_alpha, N))

for e_i, e in enumerate(epsilons):
    print("row", e_i, "testing e=", e)
    for a_i, a in enumerate(alphas):
        print("testing a=", a)
        for n in range(N):
            result = sim_instance(S, alpha=a, mrtakethatriskvalue=e, h_si=0.04, batch_run=True)
            mean_loss[e_i, a_i, n] = result["mean_loss"]
            mean_dist[e_i, a_i, n] = result["mean_dist"]
            min_loss[e_i, a_i, n]  = result["min_loss"]
            max_incd[e_i, a_i, n] = result["max_incd"]

mean_loss_grid = mean_loss.mean(axis=2) # flatten layers into avergade table
mean_dist_grid = mean_dist.mean(axis=2)
min_loss_grid  = min_loss.mean(axis=2)
max_incd_grid = max_incd.mean(axis=2) 


row 0 testing e= 0.0
testing a= 0.0
testing a= 0.1
testing a= 0.2
testing a= 0.30000000000000004
testing a= 0.4
testing a= 0.5
testing a= 0.6000000000000001
testing a= 0.7000000000000001
testing a= 0.8
testing a= 0.9
testing a= 1.0
row 1 testing e= 0.1
testing a= 0.0
testing a= 0.1
testing a= 0.2
testing a= 0.30000000000000004
testing a= 0.4
testing a= 0.5
testing a= 0.6000000000000001
testing a= 0.7000000000000001
testing a= 0.8
testing a= 0.9
testing a= 1.0
row 2 testing e= 0.2
testing a= 0.0
testing a= 0.1
testing a= 0.2
testing a= 0.30000000000000004
testing a= 0.4
testing a= 0.5
testing a= 0.6000000000000001
testing a= 0.7000000000000001
testing a= 0.8
testing a= 0.9
testing a= 1.0
row 3 testing e= 0.30000000000000004
testing a= 0.0
testing a= 0.1
testing a= 0.2
testing a= 0.30000000000000004
testing a= 0.4
testing a= 0.5
testing a= 0.6000000000000001
testing a= 0.7000000000000001
testing a= 0.8
testing a= 0.9
testing a= 1.0
row 4 testing e= 0.4
testing a= 0.0
testing a= 0.1
testi

In [ ]:
A, E = np.meshgrid(alphas, epsilons)   # both come out (n_eps, n_alpha) = (51, 51)

fig = plt.figure()
ax = fig.add_subplot(projection="3d")

surf = ax.plot_surface(A, E, mean_dist_grid, cmap="viridis")

ax.set_xlabel("alpha")
ax.set_ylabel("epsilon")
ax.set_zlabel("mean loss")
fig.colorbar(surf, label="mean loss")
plt.show()


In [24]:
S = 20000
N = 50

alphas  = np.linspace(0, 1, 11)
n_alphas = len(alphas)

mean_loss = np.empty((n_alphas, N)) # table with N layers
mean_dist = np.empty((n_alphas, N))
min_loss  = np.empty((n_alphas, N))
max_incd  = np.empty((n_alphas, N))

for alpha_i, alpha in enumerate(alphas):
    print("row", alpha_i, "testing alpha=", alpha)
    for n in range(N):
        result = sim_instance(S, alpha=alpha, hmax=2, batch_run=True)
        mean_loss[alpha_i, n] = result["mean_loss"]
        mean_dist[alpha_i, n] = result["mean_dist"]
        min_loss[alpha_i, n]  = result["min_loss"]
        max_incd[alpha_i, n] = result["max_incd"]

mean_loss_grid = mean_loss.mean(axis=1) # flatten layers into avergade table
mean_dist_grid = mean_dist.mean(axis=1)
min_loss_grid  = min_loss.mean(axis=1)
max_incd_grid = max_incd.mean(axis=1) 


row 0 testing alpha= 0.0
row 1 testing alpha= 0.1
row 2 testing alpha= 0.2
row 3 testing alpha= 0.30000000000000004
row 4 testing alpha= 0.4
row 5 testing alpha= 0.5
row 6 testing alpha= 0.6000000000000001
row 7 testing alpha= 0.7000000000000001
row 8 testing alpha= 0.8
row 9 testing alpha= 0.9
row 10 testing alpha= 1.0


In [25]:
fig, ax = plt.subplots()

line = ax.plot(alphas, mean_loss_grid, lw=1)
ax.set_xlabel("Alpha")
ax.set_ylabel("Mean Loss")

plt.show()